# Python basics

**Objective:** Combine the Python features used most often in maintainable backend services.

## Type hints

In [1]:
# Type hints document the values this function accepts and returns.
def total(prices: list[float]) -> float:
    return sum(prices)


print(total([9.99, 4.50]))

14.49


## Exceptions

In [2]:
# A domain-specific exception separates an expected business failure.
class InsufficientBalanceError(ValueError):
    pass


def withdraw(balance: float, amount: float) -> float:
    if amount > balance:
        raise InsufficientBalanceError("insufficient balance")
    return balance - amount


try:
    withdraw(50, 80)
except InsufficientBalanceError as error:
    print("Handled:", error)

Handled: insufficient balance


## Data classes

In [3]:
from dataclasses import dataclass


# Typed data: describes the customer value passed between layers.
@dataclass(frozen=True)
class User:
    id: int
    email: str


print(User(id=1, email="ada@example.com"))

User(id=1, email='ada@example.com')


## Project structure and dependencies

In [4]:
# Keep HTTP, business rules, data, and tests in clearly named modules.
layers = {
    "routers/": "HTTP endpoints",
    "services/": "business rules",
    "models.py": "data models",
    "tests/": "automated verification",
}
dependencies = {
    "runtime": ["fastapi"],
    "development": ["ruff"],
}

print(layers)
print(dependencies)

{'routers/': 'HTTP endpoints', 'services/': 'business rules', 'models.py': 'data models', 'tests/': 'automated verification'}
{'runtime': ['fastapi'], 'development': ['ruff']}


## Polished version

A service slice combines typed data, an interface, an adapter, a domain error, and business logic.

In [5]:
from dataclasses import dataclass
from typing import Optional, Protocol


@dataclass(frozen=True)
class Customer:
    id: int
    email: str


# Interface: defines what the service needs, without choosing storage.
class CustomerRepository(Protocol):
    def find_by_email(self, email: str) -> Optional[Customer]: ...
    def add(self, email: str) -> Customer: ...


# Domain error: gives a business-rule failure a meaningful name.
class DuplicateEmailError(ValueError):
    pass


# Adapter: provides the interface using an in-memory list.
# It could later be replaced by a database adapter.
class MemoryCustomerRepository:
    def __init__(self) -> None:
        self.customers: list[Customer] = []

    def find_by_email(self, email: str) -> Optional[Customer]:
        return next(
            (customer for customer in self.customers if customer.email == email),
            None,
        )

    def add(self, email: str) -> Customer:
        customer = Customer(id=len(self.customers) + 1, email=email)
        self.customers.append(customer)
        return customer


# Business logic: coordinates the rule without knowing storage details.
class CustomerService:
    def __init__(self, customers: CustomerRepository) -> None:
        # Receive any adapter that follows CustomerRepository.
        self.customers = customers

    def register(self, email: str) -> Customer:
        # Normalize first so differently formatted copies count as duplicates.
        normalized = email.strip().lower()
        if self.customers.find_by_email(normalized):
            # Enforce the business rule: customer emails must be unique.
            raise DuplicateEmailError(normalized)
        # Ask the adapter to store the valid customer.
        return self.customers.add(normalized)


# Assemble this service slice by connecting the service to one adapter.
service = CustomerService(MemoryCustomerRepository())
print(service.register("Ada@Example.com"))

Customer(id=1, email='ada@example.com')
